# Recherche de produits et produits similaires

Un catalogue de 20 000 produits d'Amazon, 150 vraies requêtes de clients, et pour chacune des jugements humains de pertinence : E (*Exact*, le produit demandé), S (*Substitute*, un substitut acceptable), C (*Complement*, un complément, par exemple un étui pour un téléphone) ou I (*Irrelevant*). Le catalogue et les requêtes sont en anglais.

1. Écrire les mesures d'évaluation
2. Une référence : BM25
3. La recherche dense, exacte puis approximative (HNSW)
4. La recherche hybride
5. Le reclassement par un cross-encodeur
6. Comparer, et lire les échecs
7. Des produits similaires

Ce notebook a besoin d'un GPU : *Exécution > Modifier le type d'exécution > GPU T4*.

## Données et modèles

In [ ]:
!wget -qO- https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh | bash -s esci-products-subset
%pip install -q bm25s faiss-cpu

In [ ]:
import time

import numpy as np
import pandas as pd
from sentence_transformers import CrossEncoder, SentenceTransformer

products = pd.read_parquet("esci-products-subset/products.parquet")
queries = pd.read_parquet("esci-products-subset/queries.parquet")
qrels = pd.read_parquet("esci-products-subset/qrels.parquet")
products["text"] = products.title + ". " + products.brand + ". " + products.bullets
print(len(products), "produits,", len(queries), "requêtes,", len(qrels), "jugements")

embedder = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", device="cuda").half()
embedder.max_seq_length = 128  # le début d'une fiche produit suffit, et l'encodage va deux fois plus vite
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", device="cuda", max_length=512)
reranker.model.half()
print("Modèles chargés")

## 1. Écrire les mesures d'évaluation

Un moteur de recherche renvoie, pour chaque requête, une liste ordonnée d'identifiants de produits. On le mesure avec :

- le **rappel à 100** : la part des produits *Exact* de la requête qu'on trouve dans les 100 premiers résultats (ce que verra une étape de reclassement)
- le **nDCG à 10** : la qualité des 10 premiers résultats (ce que voit le client). Chaque résultat rapporte un gain selon son jugement (E : 1, S : 0,1, C : 0,01, I ou non jugé : 0), divisé par $\log_2(\text{rang} + 1)$ ; la somme (DCG) est divisée par celle du meilleur classement possible (DCG idéal)

Complétez `recall_at` et `ndcg_at`, puis lancez la cellule de test. `gains[query_id]` associe à chaque produit jugé pour une requête son gain.

In [ ]:
GAIN = {"E": 1.0, "S": 0.1, "C": 0.01, "I": 0.0}
gains = {
    query_id: dict(zip(group.product_id, group.esci_label.map(GAIN)))
    for query_id, group in qrels.groupby("query_id")
}


def recall_at(ranking: list[str], query_gains: dict[str, float], k: int = 100) -> float:
    ...


def ndcg_at(ranking: list[str], query_gains: dict[str, float], k: int = 10) -> float:
    ...

### Solution

In [ ]:
def recall_at(ranking: list[str], query_gains: dict[str, float], k: int = 100) -> float:
    exact = {product for product, gain in query_gains.items() if gain == 1.0}
    return len(exact & set(ranking[:k])) / len(exact)


def ndcg_at(ranking: list[str], query_gains: dict[str, float], k: int = 10) -> float:
    dcg = sum(query_gains.get(product, 0.0) / np.log2(rank + 2) for rank, product in enumerate(ranking[:k]))
    ideal = sorted(query_gains.values(), reverse=True)[:k]
    idcg = sum(gain / np.log2(rank + 2) for rank, gain in enumerate(ideal))
    return dcg / idcg

In [ ]:
toy = {"a": 1.0, "b": 0.1, "c": 0.0, "d": 1.0}
assert recall_at(["a", "x", "d"], toy) == 1.0
assert recall_at(["a", "x", "y"], toy, k=2) == 0.5
assert abs(ndcg_at(["a", "d", "b"], toy) - 1.0) < 1e-9
assert abs(ndcg_at(["x", "a", "d"], toy) - 0.6728001281246765) < 1e-9


def evaluate(run: dict[int, list[str]]) -> pd.Series:
    """The mean recall@100 and nDCG@10 of a run: query ID -> ranked product IDs."""
    return pd.Series({
        "rappel@100": np.mean([recall_at(run[q], gains[q]) for q in run]),
        "nDCG@10": np.mean([ndcg_at(run[q], gains[q]) for q in run]),
    })


print("Mesures correctes")

## 2. Une référence : BM25

[`bm25s`](https://github.com/xhluca/bm25s) indexe un corpus pour BM25 en quelques secondes. Indexez les textes des produits (`bm25s.tokenize(textes, stopwords="en")`, puis `bm25s.BM25().index(...)`), retrouvez les 100 meilleurs produits de chaque requête (`retriever.retrieve(bm25s.tokenize(requêtes, stopwords="en"), k=100)`, qui renvoie des indices de produits et leurs scores), rangez-les dans `runs["BM25"]` sous la forme `{query_id: [product_id, ...]}`, et évaluez.

In [ ]:
runs = {}

In [ ]:
# Votre code ici

### Solution

In [ ]:
import bm25s

retriever = bm25s.BM25()
retriever.index(bm25s.tokenize(products.text.tolist(), stopwords="en"))
indices, _ = retriever.retrieve(bm25s.tokenize(queries["query"].tolist(), stopwords="en"), k=100)
runs["BM25"] = {
    query_id: products.product_id.iloc[row].tolist()
    for query_id, row in zip(queries.query_id, indices)
}
evaluate(runs["BM25"])

## 3. La recherche dense

Encodez les produits (`embedder.encode_document`, `batch_size=32`) et les requêtes (`embedder.encode_query`). Les vecteurs sont normalisés : le cosinus est un produit scalaire. Trouvez les 100 produits les plus proches de chaque requête par un produit de matrices et `np.argsort`, rangez le résultat dans `runs["dense"]` et évaluez.

In [ ]:
# Votre code ici

### Solution

In [ ]:
start = time.time()
product_vectors = embedder.encode_document(products.text.tolist(), batch_size=32, show_progress_bar=True).astype(np.float32)
query_vectors = embedder.encode_query(queries["query"].tolist()).astype(np.float32)
print(f"{time.time() - start:.0f} s pour encoder le catalogue")

scores = query_vectors @ product_vectors.T
top = np.argsort(-scores, axis=1)[:, :100]
runs["dense"] = {
    query_id: products.product_id.iloc[row].tolist()
    for query_id, row in zip(queries.query_id, top)
}
evaluate(runs["dense"])

Comparer une requête à tous les produits est exact, mais coûte trop cher à des millions de produits. Un index **HNSW** (un graphe de voisinage à plusieurs niveaux) trouve les plus proches voisins **approximativement**, beaucoup plus vite.

Construisez un index `faiss.IndexHNSWFlat(dimension, 32, faiss.METRIC_INNER_PRODUCT)`, ajoutez-y les vecteurs des produits, et cherchez les 100 plus proches voisins des requêtes (`index.search(query_vectors, 100)`). Mesurez le temps de recherche face à la recherche exacte, et la part des 100 voisins exacts que HNSW retrouve. Essayez `index.hnsw.efSearch = 16`, puis `128` : qu'est-ce que ce paramètre règle ?

In [ ]:
# Votre code ici

### Solution

In [ ]:
import faiss

index = faiss.IndexHNSWFlat(product_vectors.shape[1], 32, faiss.METRIC_INNER_PRODUCT)
index.add(product_vectors)
for ef in (16, 64, 128):
    index.hnsw.efSearch = ef
    start = time.time()
    _, approximate = index.search(query_vectors, 100)
    elapsed = time.time() - start
    overlap = np.mean([len(set(a) & set(b)) / 100 for a, b in zip(approximate, top)])
    print(f"efSearch {ef:3} : {elapsed * 1000:.0f} ms, {overlap:.1%} des voisins exacts")

start = time.time()
np.argsort(-(query_vectors @ product_vectors.T), axis=1)[:, :100]
print(f"recherche exacte : {(time.time() - start) * 1000:.0f} ms")

# efSearch est la taille de la liste de candidats explorée dans le graphe : plus grand,
# plus juste et plus lent. Sur 20 000 produits, la recherche exacte reste rapide ; HNSW
# devient indispensable à des millions de vecteurs.

## 4. La recherche hybride

La fusion par les rangs (*Reciprocal Rank Fusion*, RRF) combine plusieurs classements sans avoir à comparer leurs scores : chaque produit reçoit $\sum 1 / (60 + \text{rang})$ sur les classements où il apparaît (rang à partir de 1). Écrivez `rrf(*rankings)`, fusionnez BM25 et la recherche dense pour chaque requête dans `runs["hybride"]` (gardez les 100 premiers), et évaluez.

In [ ]:
# Votre code ici

### Solution

In [ ]:
from collections import defaultdict


def rrf(*rankings: list[str], k: int = 60) -> list[str]:
    scores = defaultdict(float)
    for ranking in rankings:
        for rank, product in enumerate(ranking, start=1):
            scores[product] += 1 / (k + rank)
    return sorted(scores, key=scores.get, reverse=True)


runs["hybride"] = {
    query_id: rrf(runs["BM25"][query_id], runs["dense"][query_id])[:100]
    for query_id in queries.query_id
}
evaluate(runs["hybride"])

# La fusion trouve plus de produits exacts dans ses 100 premiers (rappel@100) : c'est ce
# qu'exploite l'étape suivante, le reclassement. Son nDCG@10 n'est pas forcément meilleur
# que celui de la recherche dense quand BM25 est nettement plus faible.

## 5. Le reclassement par un cross-encodeur

Un cross-encodeur lit la requête et le produit **ensemble** : plus juste, mais une passe du modèle par paire. Pour les 100 premières requêtes, reclassez les 50 premiers résultats hybrides avec `reranker.predict([(requête, texte), ...], batch_size=32)`, puis ajoutez derrière les 50 suivants dans l'ordre hybride. Rangez le résultat dans `runs["hybride + reclassement"]` et comparez avec l'hybride sur les **mêmes** 100 requêtes.

In [ ]:
# Votre code ici

### Solution

In [ ]:
text_of = dict(zip(products.product_id, products.text))
query_of = dict(zip(queries.query_id, queries["query"]))
subset = queries.query_id.iloc[:100]

start = time.time()
reranked = {}
for query_id in subset:
    candidates = runs["hybride"][query_id][:50]
    scores = reranker.predict([(query_of[query_id], text_of[p]) for p in candidates], batch_size=32)
    order = [candidates[i] for i in np.argsort(-scores)]
    reranked[query_id] = order + runs["hybride"][query_id][50:]
runs["hybride + reclassement"] = reranked
print(f"{time.time() - start:.0f} s pour reclasser 100 requêtes")

pd.DataFrame({
    "hybride": evaluate({q: runs["hybride"][q] for q in subset}),
    "hybride + reclassement": evaluate(reranked),
})

## 6. Comparer, et lire les échecs

Rassemblez dans un tableau les mesures de toutes les méthodes, sur les 100 requêtes reclassées. Puis cherchez les requêtes où BM25 bat nettement la recherche dense (en nDCG@10), et l'inverse : lisez-les. Qu'ont-elles en commun ?

In [ ]:
# Votre code ici

### Solution

In [ ]:
print(pd.DataFrame({name: evaluate({q: run[q] for q in subset}) for name, run in runs.items()}).round(3))

per_query = pd.DataFrame({
    "query": [query_of[q] for q in queries.query_id],
    "BM25": [ndcg_at(runs["BM25"][q], gains[q]) for q in queries.query_id],
    "dense": [ndcg_at(runs["dense"][q], gains[q]) for q in queries.query_id],
})
per_query["écart"] = per_query.dense - per_query.BM25
print("BM25 gagne :", per_query.nsmallest(8, "écart")["query"].tolist())
print("Le dense gagne :", per_query.nlargest(8, "écart")["query"].tolist())

# BM25 gagne souvent sur les références, marques et noms exacts ; la recherche dense sur
# les requêtes formulées avec d'autres mots que les fiches produits, ou avec des fautes.

## 7. Des produits similaires

Pour la page d'un produit, on propose des produits similaires : ses plus proches voisins dans l'espace des *embeddings*, sans requête. Écrivez `similar(product_id, k=5, same_brand=False)` qui affiche le titre des `k` voisins d'un produit (sans le produit lui-même), éventuellement restreints à la même marque. Essayez sur quelques produits.

Et puisque le modèle est multilingue : cherchez dans le catalogue anglais avec des requêtes en français (« casque de moto avec écouteurs », « gamelle pour chien surélevée »).

In [ ]:
# Votre code ici

### Solution

In [ ]:
position = {product: i for i, product in enumerate(products.product_id)}


def similar(product_id: str, k: int = 5, same_brand: bool = False) -> None:
    i = position[product_id]
    scores = product_vectors @ product_vectors[i]
    scores[i] = -np.inf
    if same_brand:
        scores[products.brand.to_numpy() != products.brand[i]] = -np.inf
    print("Produit :", products.title[i], "| marque :", products.brand[i])
    for j in np.argsort(-scores)[:k]:
        if np.isfinite(scores[j]):
            print(f"  {scores[j]:.3f} {products.title[j][:100]}")


# Des produits dont la marque a d'autres produits au catalogue.
brands = products.brand[products.brand != ""].value_counts()
candidates = products[products.brand.isin(brands[brands >= 5].index)]
for product_id in candidates.product_id.sample(3, random_state=0):
    similar(product_id)
    similar(product_id, same_brand=True)

for french in ("casque de moto avec écouteurs", "gamelle pour chien surélevée"):
    scores = product_vectors @ embedder.encode_query(french).astype(np.float32)
    print(french, "->", products.title.iloc[np.argsort(-scores)[:3]].str[:80].tolist())